<a href="https://colab.research.google.com/github/mohammedAlkhuzaie/Suha-Ali-Salman/blob/main/Car_configuration_Suha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Car Configuration
==================

Problem
-------
Cars are configured step by step from a set of independent option groups
(engine, transmission, interior, exterior, safety). Configurations are
optional and vary by model -- a given model may not offer every option --
so the construction process needs to be flexible while still guaranteeing
that whatever comes out at the end is a *valid* car.

Design
------
This uses the **Builder** pattern:

- `Car` is the immutable product: a plain data holder for the finished
  configuration.
- `CarBuilder` exposes one fluent method per configurable step
  (`set_engine`, `set_transmission`, `add_interior_feature`, ...). Each
  method returns `self`, so steps can be chained in any order and any subset
  of them can be skipped.
- Each model defines its own allowed options and its own required options
  via a `CarModelSpec`, so "not all cars have the same set of available
  options" is enforced per-model rather than hard-coded into the builder.
- `build()` performs validation (required options present, chosen options
  legal for this model) and only then returns a `Car`. This guarantees the
  final object is always valid and ready for ordering -- an invalid
  configuration simply cannot produce a `Car`.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set


class InvalidConfigurationError(ValueError):
    """Raised when build() is called on a configuration that is incomplete
    or that selects an option the model does not support."""


@dataclass(frozen=True)
class CarModelSpec:
    """Describes what a given model allows and requires."""

    name: str
    allowed_engines: Set[str]
    allowed_transmissions: Set[str]
    allowed_interior_features: Set[str]
    allowed_exterior_options: Set[str]
    allowed_safety_features: Set[str]
    required_engine: bool = True
    required_transmission: bool = True
    required_safety_features: Set[str] = field(default_factory=set)


@dataclass(frozen=True)
class Car:
    """The finished, valid, order-ready product."""

    model: str
    engine: Optional[str]
    transmission: Optional[str]
    interior_features: List[str]
    exterior_options: Dict[str, str]
    safety_features: List[str]

    def summary(self) -> str:
        lines = [f"Car order: {self.model}"]
        lines.append(f"  Engine: {self.engine or '—'}")
        lines.append(f"  Transmission: {self.transmission or '—'}")
        lines.append(f"  Interior: {', '.join(self.interior_features) or 'none'}")
        lines.append(
            f"  Exterior: {', '.join(f'{k}={v}' for k, v in self.exterior_options.items()) or 'none'}"
        )
        lines.append(f"  Safety: {', '.join(self.safety_features) or 'none'}")
        return "\n".join(lines)


class CarBuilder:
    """Step-by-step, fluent, flexible builder. Any step may be skipped;
    build() validates the result against the model's spec before returning
    a Car."""

    def __init__(self, spec: CarModelSpec) -> None:
        self._spec = spec
        self._engine: Optional[str] = None
        self._transmission: Optional[str] = None
        self._interior_features: List[str] = []
        self._exterior_options: Dict[str, str] = {}
        self._safety_features: List[str] = []

    def set_engine(self, engine: str) -> "CarBuilder":
        if engine not in self._spec.allowed_engines:
            raise InvalidConfigurationError(
                f"Engine '{engine}' is not offered on {self._spec.name}. "
                f"Choose from {sorted(self._spec.allowed_engines)}."
            )
        self._engine = engine
        return self

    def set_transmission(self, transmission: str) -> "CarBuilder":
        if transmission not in self._spec.allowed_transmissions:
            raise InvalidConfigurationError(
                f"Transmission '{transmission}' is not offered on {self._spec.name}. "
                f"Choose from {sorted(self._spec.allowed_transmissions)}."
            )
        self._transmission = transmission
        return self

    def add_interior_feature(self, feature: str) -> "CarBuilder":
        if feature not in self._spec.allowed_interior_features:
            raise InvalidConfigurationError(
                f"Interior feature '{feature}' is not offered on {self._spec.name}."
            )
        if feature not in self._interior_features:
            self._interior_features.append(feature)
        return self

    def set_exterior_option(self, key: str, value: str) -> "CarBuilder":
        allowed = self._spec.allowed_exterior_options
        if key not in allowed:
            raise InvalidConfigurationError(
                f"Exterior option '{key}' is not offered on {self._spec.name}."
            )
        self._exterior_options[key] = value
        return self

    def add_safety_feature(self, feature: str) -> "CarBuilder":
        if feature not in self._spec.allowed_safety_features:
            raise InvalidConfigurationError(
                f"Safety feature '{feature}' is not offered on {self._spec.name}."
            )
        if feature not in self._safety_features:
            self._safety_features.append(feature)
        return self

    def build(self) -> Car:
        if self._spec.required_engine and self._engine is None:
            raise InvalidConfigurationError(f"{self._spec.name} requires an engine to be selected.")
        if self._spec.required_transmission and self._transmission is None:
            raise InvalidConfigurationError(f"{self._spec.name} requires a transmission to be selected.")
        missing_required_safety = self._spec.required_safety_features - set(self._safety_features)
        if missing_required_safety:
            raise InvalidConfigurationError(
                f"{self._spec.name} requires safety features {sorted(missing_required_safety)} "
                f"which have not been selected."
            )
        return Car(
            model=self._spec.name,
            engine=self._engine,
            transmission=self._transmission,
            interior_features=list(self._interior_features),
            exterior_options=dict(self._exterior_options),
            safety_features=list(self._safety_features),
        )


# Example model specs -- each model owns its own allowed/required options,
# which is how "not all cars have the same set of available options" is
# satisfied without special-casing inside the builder itself.
SEDAN_SPEC = CarModelSpec(
    name="Sedan LX",
    allowed_engines={"V6"},
    allowed_transmissions={"automatic", "manual"},
    allowed_interior_features={"leather seats", "GPS", "sound system"},
    allowed_exterior_options={"color", "rims"},
    allowed_safety_features={"ABS", "airbags", "rear camera"},
    required_safety_features={"ABS", "airbags"},
)

SUV_SPEC = CarModelSpec(
    name="SUV XT",
    allowed_engines={"V6", "V8"},
    allowed_transmissions={"automatic"},
    allowed_interior_features={"leather seats", "GPS", "sound system"},
    allowed_exterior_options={"color", "rims", "sunroof"},
    allowed_safety_features={"ABS", "airbags", "rear camera"},
    required_safety_features={"ABS", "airbags", "rear camera"},
)


if __name__ == "__main__":
    car = (
        CarBuilder(SEDAN_SPEC)
        .set_engine("V6")
        .set_transmission("automatic")
        .add_interior_feature("GPS")
        .set_exterior_option("color", "black")
        .add_safety_feature("ABS")
        .add_safety_feature("airbags")
        .build()
    )
    print(car.summary())


Car order: Sedan LX
  Engine: V6
  Transmission: automatic
  Interior: GPS
  Exterior: color=black
  Safety: ABS, airbags
